# Fine-tuning Llama 3.1 for Egyptian Education Chatbot 🇪🇬🎓

This notebook covers the full pipeline to fine-tune **Llama 3.1 8B Instruct** on an **Egyptian Arabic dataset** to create a specialized tutor/teacher chatbot.

**Features:**
- Uses **Unsloth** for 2x faster training and 60% less memory.
- Quantization (4-bit) to run on free Kaggle/Colab GPUs (T4).
- Custom data loader for the Egyptian Dialogues JSON.
- Exports to **GGUF** format for easy use in GUIs (like LM Studio, Ollama, or Python apps).

In [ ]:
!pip install -q --force-reinstall protobuf==4.25.3
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers peft accelerate bitsandbytes


import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 6.3 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 4.25.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you h

In [ ]:
import json
import torch
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

max_seq_length = 768
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-14 17:46:30.763491: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765734390.957909     180 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765734391.008853     180 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.12.5 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
import os
dataset_dir = "/kaggle/input/egyptian-dialogues"

# Find the JSON file automatically
json_file = None
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
            print(f"Found JSON file: {json_file}")
            break
    if json_file:
        break

if json_file is None:
    raise FileNotFoundError("No JSON file found in the dataset directory!")

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} dialogues successfully!")

# Format as chat conversations
formatted = []
for item in data:
    convo = f"موضوع الدرس: {item['subject']}\n\n"
    for turn in item['dialogue']:
        role = "المعلم" if turn['role'] == "teacher" else "الطالب"
        convo += f"{role}: {turn['text']}\n"
    convo += "<|end_of_text|>"  # Optional EOS
    formatted.append(convo)

dataset = Dataset.from_list([{"text": t} for t in formatted])
dataset = dataset.train_test_split(test_size=0.1)

print(f"Dataset ready: {len(dataset['train'])} train examples, {len(dataset['test'])} validation examples")

Found JSON file: /kaggle/input/egyptian-dialogues/egyptian_dialogues.json
Loaded 551 dialogues successfully!
Dataset ready: 495 train examples, 56 validation examples


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=20,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        output_dir="/kaggle/working/egyptian-chatbot-unsloth",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()

trainer.save_model("/kaggle/working/egyptian-chatbot-final")
tokenizer.save_pretrained("/kaggle/working/egyptian-chatbot-final")
print("Training complete! Model saved.")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/495 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/56 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 495 | Num Epochs = 5 | Total steps = 155
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 167,772,160 of 8,198,033,408 (2.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,1.493600,1.552549
100,1.145800,1.452595
150,0.958000,1.458749


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training complete! Model saved.


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "/kaggle/working/egyptian-chatbot-final",
    dtype=None,
    load_in_4bit=False,
)

merged_model = model.merge_and_unload()
merged_model.save_pretrained("/kaggle/working/egyptian-chatbot-merged")
tokenizer.save_pretrained("/kaggle/working/egyptian-chatbot-merged")
FastLanguageModel.for_inference(merged_model)

prompt = """موضوع الدرس: رياضيات
الطالب: يا أستاذ، مش فاهم قاعدة السلسلة خالص.
المعلم:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = merged_model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

موضوع الدرس: رياضيات
الطالب: يا أستاذ، مش فاهم قاعدة السلسلة خالص.
المعلم: ههه، طب بص، هي سهلة أوي بس محتاجة تركيز. بص، تخيل إنك عايز تقسم حاجة كبيرة على حاجة صغيرة، صح؟
الطالب: ماشي، فهمت الحتة دي.
المعلم: تمام. يعني لو عندك دالة صغيرة في دالة كبيرة، زي (3 + س^2) في (س - 1)، بنعمل إيه؟
الطالب: بنزل السمايل؟
المعلم: أيوه! بس مش بالظبط كده. بنزل السمايل، وبعدين نضرب في التكامل بتاع اللي جوه القوس. يعني تبقى 1 على (س - 1).
الطالب: آه! يعني (3 + س^2) في (س - 1) تبقى 1 على (س - 1) في تكامل (3 +
